# 03 — Задать вопрос: Qdrant → reranker → ответ LLM

Один проход по вопросу: USER2-base забирает из Qdrant **30** кандидатов, русскоязычный кросс-энкодер `sshalimov04/ru-reranker-edge-150m` переупорядочивает их, в контекст LLM уходят **10** лучших. В ответе LLM должна опираться только на эти фрагменты, ссылаться на `[S1]…[S10]` и прямо сказать, если источников недостаточно.

Модель не может гарантировать отсутствие ошибок, поэтому ниже отдельно показаны 10 источников, оценки обоих этапов поиска и ссылки на вложения. Прочитайте их, если ответ важен.

In [ ]:
# У этого ноутбука отдельное ядро: свежему реранкеру нужна Transformers 5.16.1,
# а предыдущие ноутбуки USER2 используют Transformers 4.x.
# При необходимости в этом ядре:
# %pip install 'transformers==5.16.1' 'qdrant-client>=1.19' pandas requests


In [ ]:
from pathlib import Path
import requests
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display, Markdown
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from qdrant_client import QdrantClient
import httpx
from huggingface_hub.utils._http import set_client_factory, hf_request_event_hook

# Убираем Brotli из HTTP Accept-Encoding: это предотвращает ошибку декодирования
# больших файлов модели на этой машине.
set_client_factory(
    lambda: httpx.Client(
        headers={"accept-encoding": "gzip, deflate"},
        event_hooks={"request": [hf_request_event_hook]},
        follow_redirects=True,
        timeout=None,
    )
)

# Поменяйте QUESTION, затем выполните ноутбук сверху вниз.
QUESTION = "Что известно о контрольной работе по теории вероятностей «Хайзенберг»?"

QDRANT_URL = "http://localhost:6333"
COLLECTION = "telegram_keep_user2_350tok_50overlap"
EMBEDDING_MODEL = "deepvk/USER2-base"
RERANKER_MODEL = "sshalimov04/ru-reranker-edge-150m"
OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "qwen3.5:4b"
CANDIDATE_COUNT = 30
FINAL_CONTEXT_COUNT = 10
RERANKER_MAX_LENGTH = 512
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)

assert QUESTION.strip(), "Введите вопрос в QUESTION"
client = QdrantClient(url=QDRANT_URL, timeout=30)
collection_names = {c.name for c in client.get_collections().collections}
assert COLLECTION in collection_names, (
    f"Коллекция {COLLECTION} не найдена; сначала выполните 01_qdrant_ingest_user2.ipynb"
)
print("Вопрос:", QUESTION)
print("Устройство моделей:", DEVICE)

## 1. Семантический поиск: 30 кандидатов

In [ ]:
# USER2-base в Transformers: префикс search_query + mean pooling + L2-нормализация.
embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
embedding_model = AutoModel.from_pretrained(EMBEDDING_MODEL).to(DEVICE).eval()
encoded_query = embedding_tokenizer(
    ["search_query: " + QUESTION], padding=True, truncation=True, return_tensors="pt"
).to(DEVICE)
with torch.inference_mode():
    output = embedding_model(**encoded_query)
    mask = (
        encoded_query["attention_mask"]
        .unsqueeze(-1)
        .expand(output.last_hidden_state.size())
        .float()
    )
    pooled = (output.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(
        min=1e-9
    )
    query_vector = F.normalize(pooled, p=2, dim=1)[0].cpu().tolist()

search_result = client.query_points(
    collection_name=COLLECTION,
    query=query_vector,
    limit=CANDIDATE_COUNT,
    with_payload=True,
).points
if not search_result:
    raise ValueError("Qdrant не вернул кандидатов для вопроса")
print(f"Кандидатов из Qdrant: {len(search_result)}")

## 2. Повторное ранжирование: оставить 10 фрагментов

In [ ]:
# Последняя версия русского кросс-энкодера поддерживает Transformers 5.x.
rerank_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL)
reranker = (
    AutoModelForSequenceClassification.from_pretrained(RERANKER_MODEL).to(DEVICE).eval()
)
passages = [hit.payload.get("text", "") for hit in search_result]
rerank_scores = []
with torch.inference_mode():
    for start in range(0, len(passages), 8):
        batch_passages = passages[start : start + 8]
        inputs = rerank_tokenizer(
            [QUESTION] * len(batch_passages),
            batch_passages,
            padding=True,
            truncation=True,
            max_length=RERANKER_MAX_LENGTH,
            return_tensors="pt",
        ).to(DEVICE)
        logits = reranker(**inputs).logits.squeeze(-1)
        rerank_scores.extend(logits.float().cpu().tolist())

ranked = sorted(
    zip(search_result, rerank_scores), key=lambda pair: float(pair[1]), reverse=True
)[:FINAL_CONTEXT_COUNT]
selected = []
for source_no, (hit, score) in enumerate(ranked, 1):
    p = hit.payload
    selected.append(
        {
            "source_id": f"S{source_no}",
            "rerank_score": float(score),
            "semantic_score": float(hit.score),
            "text": p.get("text", ""),
            "date": p.get("date", ""),
            "chat_name": p.get("chat_name", ""),
            "sender": p.get("sender", ""),
            "record_id": p.get("record_id", ""),
            "file_paths": p.get("file_paths", []),
            "pdf_paths": p.get("pdf_paths", []),
        }
    )
reranked_df = pd.DataFrame(selected)
print(f"Передаём LLM {len(selected)} из {len(search_result)} результатов.")
display(
    reranked_df[
        [
            "source_id",
            "rerank_score",
            "semantic_score",
            "date",
            "chat_name",
            "sender",
            "text",
        ]
    ]
)

## 3. Объединить топ-10 в контекст и спросить LLM

In [ ]:
context = "\n\n".join(
    f"[{s['source_id']}] Дата: {s['date']} | Чат: {s['chat_name']} | Автор: {s['sender']}\n{s['text']}"
    for s in selected
)

system_prompt = (
    "Ты отвечаешь на вопрос пользователя только по предоставленным фрагментам переписки. "
    "Не используй знания из памяти и не додумывай отсутствующие факты. "
    "Каждое фактическое утверждение подкрепляй ссылкой на источник в формате [S1]. "
    "Если контекста недостаточно, источники противоречат друг другу или ответ нельзя установить, "
    "прямо скажи: «В найденных сообщениях недостаточно данных, чтобы ответить уверенно». "
    "Не превращай предположение в факт. Пиши по-русски и кратко."
)
user_prompt = f"Вопрос: {QUESTION}\n\nНайденные источники:\n{context}\n\nДай ответ с цитатами вида [S1]."

response = requests.post(
    OLLAMA_URL,
    json={
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": False,
        "think": False,  # для Qwen в Ollama отключает внутренний reasoning, оставляя финальный ответ
        "options": {"temperature": 0, "num_predict": 700, "num_ctx": 8192, "seed": 42},
    },
    timeout=300,
)
response.raise_for_status()
answer = response.json()["message"]["content"].strip()
if not answer:
    raise RuntimeError(
        "LLM вернула пустой ответ. Проверьте модель Ollama и настройки запроса."
    )
display(Markdown("## Ответ\n\n" + answer))

## Источники ответа

Проверьте эти фрагменты, особенно если ответ затрагивает сроки, требования или точные числа.

In [ ]:
def local_link(path):
    p = Path(path)
    return f"[{p.name}](<{p.resolve()}>)"


for s in selected:
    files = [local_link(p) for p in s["file_paths"] if p]
    file_line = ("\n\nВложения: " + ", ".join(files)) if files else ""
    display(
        Markdown(
            f"**[{s['source_id']}]** · rerank={s['rerank_score']:.3f} · vector={s['semantic_score']:.3f} · "
            f"{s['date']} · {s['chat_name']} · {s['sender']}\n\n{s['text']}{file_line}"
        )
    )

# Удобные результаты доступны в переменных answer и reranked_df.